<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-01-setup-and-iam/lesson-1.2-iam-security/notebooks/GCP_Capstone_1.2_IAM_Security.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.2 IAM & Security for GenAI Projects
**Netsetos GenAI Engineering — GCP Capstone**

Hands-on: Create dedicated SAs, store secrets, verify ADC, query audit logs.


## Cell 0: Install Dependencies


In [ ]:
!pip install -q google-genai==2.21.0 google-cloud-secret-manager


## Cell 1: Authenticate & Set Project


In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS
!gcloud config set project {PROJECT_ID}


## Cell 2: Inspect the Default Service Account


In [ ]:
# Find the default compute SA and its dangerous Editor role
!gcloud iam service-accounts list --format='table(email,displayName)'
# Roles on the default SA (guarded: a brand-new project may not have it yet)
roles = get_ipython().getoutput(
    f"gcloud projects get-iam-policy {PROJECT_ID} --flatten='bindings[].members' "
    "--filter='bindings.members:compute@developer' --format='value(bindings.role)'")
if not roles:
    print('Default compute SA not found yet - enable compute.googleapis.com (lesson 1.1) and re-run.')
else:
    print('\n--- Roles on default SA ---')
    print('\n'.join(roles))   # expect roles/editor  <-- THIS IS THE PROBLEM


## Cell 3: Create Dedicated Service Accounts


In [ ]:
# Create the app SA with least-privilege intent. Verify-then-create: on the demo project (and on
# any re-run) the accounts exist already, and a bare create prints ALREADY_EXISTS in red.
!gcloud iam service-accounts describe sa-documind-app@{PROJECT_ID}.iam.gserviceaccount.com >/dev/null 2>&1 \
  || gcloud iam service-accounts create sa-documind-app --display-name='DocuMind Application SA' --description='Runs Gemini chatbot on Cloud Run'

# A separate SA for the CI/CD pipeline (build, push, release, deploy-as). On the demo project this
# is the kit's real pipeline identity (sa.tf): the same name, created by Terraform.
!gcloud iam service-accounts describe sa-documind-cicd@{PROJECT_ID}.iam.gserviceaccount.com >/dev/null 2>&1 \
  || gcloud iam service-accounts create sa-documind-cicd --display-name='DocuMind CI/CD SA' --description='Cloud Build deployments'

# Verify
!gcloud iam service-accounts list --filter='email:sa-documind' --format='table(email,displayName)'


## Cell 4: Grant Exactly 4 IAM Roles


In [ ]:
SA = f'sa-documind-app@{PROJECT_ID}.iam.gserviceaccount.com'

roles = [
    'roles/aiplatform.user',
    'roles/datastore.user', 
    'roles/secretmanager.secretAccessor',
    'roles/storage.objectViewer'
]

for role in roles:
    !gcloud projects add-iam-policy-binding {PROJECT_ID} \
      --member='serviceAccount:{SA}' --role='{role}' --quiet
    print(f'  Granted: {role}')

# Verify
print('\n--- Roles on sa-documind-app ---')
!gcloud projects get-iam-policy {PROJECT_ID} \
  --flatten='bindings[].members' \
  --filter='bindings.members:sa-documind-app' \
  --format='table(bindings.role)'


## Cell 4b: Grant Exactly 4 IAM Roles to the CI/CD SA

In [ ]:
CICD_SA = f'sa-documind-cicd@{PROJECT_ID}.iam.gserviceaccount.com'

# One role per pipeline stage - and nothing the app SA has
cicd_roles = [
    'roles/cloudbuild.builds.editor',   # build the container image (12.2)
    'roles/artifactregistry.writer',    # push the image to Artifact Registry
    'roles/clouddeploy.releaser',       # create Cloud Deploy releases (dev -> prod)
]

for role in cicd_roles:
    !gcloud projects add-iam-policy-binding {PROJECT_ID} \
      --member='serviceAccount:{CICD_SA}' --role='{role}' --quiet
    print(f'  Granted: {role}')

# actAs on the ONE service account the pipeline deploys as - never at project scope, where it
# would let the pipeline act as every service account in the project, present and future. The
# kit's sa.tf refuses exactly that and grants it per runtime account (12.7); on the demo project
# the account below is the kit's real CI/CD identity, so this is not a drill.
APP_SA = f'sa-documind-app@{PROJECT_ID}.iam.gserviceaccount.com'
!gcloud iam service-accounts add-iam-policy-binding {APP_SA} \
  --member='serviceAccount:{CICD_SA}' --role='roles/iam.serviceAccountUser' --quiet
print('  Granted: roles/iam.serviceAccountUser - on sa-documind-app only')

# Verify - the three project roles (on the demo project the kit adds clouddeploy.jobRunner and run.developer)
print('\n--- Roles on sa-documind-cicd ---')
!gcloud projects get-iam-policy {PROJECT_ID} \
  --flatten='bindings[].members' \
  --filter='bindings.members:sa-documind-cicd' \
  --format='table(bindings.role)'

# No key file for this SA, ever: in 12.7 keyless Workload Identity Federation replaces
# service-account keys - GitHub Actions gets a 10-minute token by OIDC and impersonates this SA.

## Cell 5: Create Secrets in Secret Manager


In [ ]:
# Create two secrets - verify-then-create, so a re-run does not print ALREADY_EXISTS
!gcloud secrets describe gemini-config >/dev/null 2>&1 || echo -n 'test-gemini-key-12345' | gcloud secrets create gemini-config --data-file=- --replication-policy=automatic --labels='env=dev'
!gcloud secrets describe db-password >/dev/null 2>&1 || echo -n 'super-secure-db-pass' | gcloud secrets create db-password --data-file=- --replication-policy=automatic

# List secrets
!gcloud secrets list --format='table(name,labels)'


## Cell 6: Access Secrets from Python


In [ ]:
from google.cloud import secretmanager

def get_secret(secret_id, version='latest'):
    client = secretmanager.SecretManagerServiceClient()
    name = f'projects/{PROJECT_ID}/secrets/{secret_id}/versions/{version}'
    response = client.access_secret_version(request={'name': name})
    return response.payload.data.decode('UTF-8')

# Test both secrets
for sid in ['gemini-config', 'db-password']:
    val = get_secret(sid)
    print(f'  {sid}: {val[:4]}****')


## Cell 7: Test Gemini Call - as yourself

This call runs under YOUR Application Default Credentials (the Colab sign-in), not as sa-documind-app: the service account is what Cloud Run will run as in 12.2, and 1.2's impersonation flow (page, Step 6) is how a person borrows it deliberately.


In [ ]:
from google import genai

client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Explain what IAM means in cloud security in 2 sentences.'
)
print('Response:', response.text)
print(f'Tokens: {response.usage_metadata.total_token_count}')


## Cell 8: Query Audit Logs


In [ ]:
# See who made API calls recently
!gcloud logging read \
  'logName:"cloudaudit.googleapis.com"' \
  --project={PROJECT_ID} --freshness=1d --limit=5 \
  --format='table(timestamp,protoPayload.methodName,protoPayload.authenticationInfo.principalEmail)'


## Cell 9: Security Audit Script


In [ ]:
import subprocess, json

result = subprocess.run(
    ['gcloud', 'projects', 'get-iam-policy', PROJECT_ID, '--format=json'],
    capture_output=True, text=True
)
policy = json.loads(result.stdout)

DANGEROUS = {'roles/editor', 'roles/owner'}
sa_roles = {}

for binding in policy.get('bindings', []):
    role = binding['role']
    for member in binding.get('members', []):
        if member.startswith('serviceAccount:'):      # a deleted:serviceAccount:... member is not a row
            sa = member.split(':', 1)[1]
            sa_roles.setdefault(sa, []).append(role)

print('\n Security Audit Report')
print('=' * 60)
alerts = 0
for sa, roles in sorted(sa_roles.items()):
    has_danger = any(r in DANGEROUS for r in roles)
    icon = '🔴' if has_danger else '🟢'
    print(f'\n{icon} {sa}')
    for r in roles:
        flag = ' ⚠ OVERPRIVILEGED' if r in DANGEROUS else ''
        print(f'   {r}{flag}')
    if has_danger:
        alerts += 1

print(f'\n Summary: {len(sa_roles)} SAs, {alerts} with dangerous roles')
if alerts:
    print(' ACTION REQUIRED: Remove Editor/Owner from service accounts!')
else:
    print(' All service accounts follow least privilege.')


## ✅ Security Setup Complete!

- ✅ Inspected dangerous default SA
- ✅ Created dedicated sa-documind-app with 4 roles
- ✅ Created sa-documind-cicd with its own 4 CI/CD roles (keyless WIF in 12.7)
- ✅ Stored secrets in Secret Manager
- ✅ Accessed secrets from Python
- ✅ Queried audit logs
- ✅ Built security audit script

**Next: Lesson 1.3 — Your First Gemini Call (Deep Dive)**
